In [1]:
import os, pathlib
print("CWD:", os.getcwd())
print("Contenido:", list(pathlib.Path(".").iterdir()))


CWD: /workspaces/TFG_Spain-s_Migratory_Flow
Contenido: [PosixPath('.venv'), PosixPath('notebooks'), PosixPath('trigger.py'), PosixPath('.gitignore'), PosixPath('config'), PosixPath('.github'), PosixPath('collectors'), PosixPath('.env'), PosixPath('fix_chdir.py'), PosixPath('run_insights.py'), PosixPath('requirements.txt'), PosixPath('data'), PosixPath('scheduler.py'), PosixPath('__pycache__'), PosixPath('README.md'), PosixPath('output'), PosixPath('.gitattributes'), PosixPath('prueba.ipynb'), PosixPath('.git')]


In [2]:
import os
from pathlib import Path
import pandas as pd

# Buscar la raíz del repo subiendo hasta encontrar requirements.txt
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if not (ROOT / "requirements.txt").exists():
    raise FileNotFoundError(
        f"No se encontró requirements.txt subiendo desde {Path.cwd()}\n"
        "Asegúrate de ejecutar desde dentro del repo."
    )

os.chdir(ROOT)
print("ROOT:", ROOT)

master_path = ROOT / "output" / "merged" / "master_provincia_anio.parquet"
print(master_path, "exists:", master_path.exists())

df = pd.read_parquet(master_path)
print(df.shape)

ROOT: /workspaces/TFG_Spain-s_Migratory_Flow
/workspaces/TFG_Spain-s_Migratory_Flow/output/merged/master_provincia_anio.parquet exists: True
(2184, 15)


In [4]:
# Ya tienes df y grp definidos

cols = [c for c in df.columns if c not in ["provincia","anyo"]]

# nº de valores distintos dentro de cada grupo (provincia, anyo) y columna
nuniq = grp[cols].nunique()

# 1) Máximo nº de valores distintos por columna (si todo es 1, perfecto)
max_nuniq = nuniq.max(axis=0)
print("max nº de valores distintos por columna dentro de cada clave:")
print(max_nuniq.sort_values(ascending=False))

# 2) Columnas conflictivas: alguna vez hay >1 valor dentro del mismo prov-anio
conflict_cols = max_nuniq[max_nuniq > 1]
print("\ncolumnas con >1 valor distinto dentro de alguna clave:")
print(conflict_cols)

max nº de valores distintos por columna dentro de cada clave:
poblacion_provincia        3
compraventas_per_capita    3
saldo_neto                 1
salidas                    1
entradas                   1
delta_saldo_yoy            1
tasa_atraccion             1
compraventas_vivienda      1
ranking_atraccion_anio     1
viajeros                   1
pernoctaciones             1
cob_100mbps_pct            1
ranking_anio               1
dtype: int64

columnas con >1 valor distinto dentro de alguna clave:
poblacion_provincia        3
compraventas_per_capita    3
dtype: int64


In [5]:
df_dedup = df.drop_duplicates(subset=["provincia","anyo"]).copy()
print(df_dedup.shape)   # debería ser (728, 15)

df_dedup.to_parquet("output/merged/master_provincia_anio_dedup.parquet", index=False)

(728, 15)
